# Assessing an Observed Galaxy

This notebook prepares a BayMorph analysis of one observed galaxy with the
observation advisor. The advisor reads the FITS header, you confirm the
target, the advisor prepares the image and builds its PSF, and it proposes a
complete configuration: every prior of every model you choose, with where it
comes from. You review the proposal and write one YAML configuration.

It does not run the fit: `02B_Observed_Fit.ipynb` reads the YAML, reuses the
prepared image and the PSF, and fits each model. The YAML is the contract
between the two notebooks.

BayMorph never derives priors from the image it will fit. The priors come
from the header, from physics and from the fixed inputs of the observation
model, the sky scale and the PSF. The quantities measured on the target are
diagnostics.

In [ ]:
from pathlib import Path

import yaml

from baymorph import ObservationAdvisor

## Observation

Relative paths are resolved from the project root, so the notebook also works
from the `notebooks` directory. Replace the FITS path with your own image.
`analysis_name` names both the YAML and the run directory.

In [ ]:
fits_path = Path("data/galaxy_r.fits")
analysis_name = "galaxy_r"
config_path = Path("configs") / f"{analysis_name}.yaml"

advisor = ObservationAdvisor.from_project(fits_path)

print(f"FITS: {advisor.fits_path}")
print(f"Project root: {advisor.project_root}")

## What the header says

The advisor reads the header through the conventions of the telescopes and
pipelines it knows, and through the FITS standard otherwise. Each row gives a
quantity, its value and unit, its status, and the keyword and convention it
comes from. The status says how far the value can be trusted:

- `declared`: you declared it, through `calibration` here or in
  `data.calibration` of a configuration;
- `explicit`: the header itself declares the value and its unit;
- `convention`: a convention of the instrument or pipeline documents its
  meaning and unit, or states the value;
- `derived`: computed from other header content, such as the pixel scale
  from the WCS;
- `ambiguous`: present, but its unit or meaning is not established;
- `conflicting`: present under several keywords, or read by several
  conventions, with different values.

The last two are shown as written and never used: where they matter, you
declare the value instead, for example
`ObservationAdvisor.from_project(fits_path, calibration={"zeropoint": 25.9})`,
and the configuration keeps it in `data.calibration`. A unit is never
guessed from a value. The zeropoint turns magnitudes into fluxes for the
flux priors, and the pixel scale turns arcseconds into pixels for the
lengths. Without a zeropoint the flux priors start from the flux of a point
source at signal-to-noise 1 on the prepared sky, in image units; without a
pixel scale the lengths keep the BayMorph defaults in pixels.

For a drizzled ACS image whose header gives no usable zeropoint, the advisor
asks the ACS Zeropoint Calculator of STScI once, and the report ends with
what it asked and answered; the configuration then holds the zeropoint, so
the fit never goes online. `ObservationAdvisor.from_project(fits_path,
online=False)` asks nothing.

In [ ]:
print(advisor.header_report())

## Target

Click the centre of the target galaxy, then press **Confirm center**. The fit
samples the centre within a window about this position, three times the FWHM
of the PSF, so it must be the centre; the proposal stops when the centre
measured on the target lies outside that window. BayMorph uses
zero-based pixel coordinates: `x` is the NumPy column and `y` the NumPy row.
A candidate position is never accepted implicitly.

Without the interactive widgets, set the centre explicitly instead with
`advisor.set_target_position(x, y)`.

In [ ]:
advisor.select_target(
    title="Select the target centre",
)

## Image diagnostics

The summary gives scales measured over the whole image and diagnostics of the
confirmed target, measured on its segment: its flux, half-light radius, axis
ratio and position angle. Use them to judge whether a proposed prior is
sensible, never to set its bounds.

In [ ]:
if advisor.target_position is None:
    raise ValueError(
        "Confirm the target centre in the selector before continuing."
    )

advisor.inspect()

print(advisor.summary())

## Preparation and PSF

The sky scale, the cutout and the PSF are fixed inputs of the observation
model: they come before any prior. The advisor runs the preparation and the
PSF stages now and writes their products to the run directory
`runs/<analysis_name>`; 02B reuses them instead of building them again. This
cell can take a few minutes.

These options shape the stages, and they are part of the scientific problem:

- `cutout_mode`: `auto` sizes the cutout from the extent of the galaxy and can
  be large for an extended source; `fixed` uses a square of `fixed_size`
  pixels. Preparation rejects a cutout that does not contain the sky plateau
  around the galaxy plus a margin.
- `psf_max_peak`: the brightest peak pixel accepted for a star of the
  empirical PSF, in image units. Left at `None`, the advisor takes half the
  saturation level of the header when its unit is established; otherwise
  BayMorph uses a default that is not scaled to the image, and the proposal
  warns.
- `preparation`: optional preparation settings, for example
  `{"noise": {"sky_correlation": "field_separated"}}` for a sky that holds a
  large-scale field.

Running the cell again reuses the products. To change an option once the
products exist, choose a new `analysis_name`: products are never rewritten
in place.

In [ ]:
stage_options = {
    "object_name": analysis_name,
    "cutout_mode": "auto",
    "fixed_size": None,
    "psf_max_peak": None,
    "preparation": None,
}

fixed_inputs = advisor.measure_fixed_inputs(**stage_options)

rows, columns = fixed_inputs.cutout_shape

print(f"Prepared sky scale: {fixed_inputs.sky_scale:.4g}")
print(f"Cutout: {columns} x {rows} pixels")
print(
    f"PSF FWHM: {fixed_inputs.psf_fwhm_pixels:.3g} pixels "
    f"({fixed_inputs.psf_fwhm_source})"
)
print(f"Products: {fixed_inputs.run_directory}")

## Models

Choose the models to fit. BayMorph distributes no validated classifier that
ranks the models for this galaxy, so the choice is yours, and it is not a
Bayesian preference. 02B fits each model separately, on the same prepared
image, PSF and likelihood, and gives each its posterior and its evidence.

| Model | Components | Parameters | Use |
| --- | --- | --- | --- |
| `single_sersic` | one Sérsic | 7 | the baseline description of a galaxy |
| `double_sersic` | two labelled Sérsic components, shared centre | 12 | a galaxy whose profile one Sérsic cannot describe |
| `sersic_point_source` | Sérsic and unresolved source, shared centre | 8 | a galaxy with an unresolved nucleus |
| `bulge_disk` | Sérsic bulge and exponential disc, shared centre | 11 | a bulge and disc decomposition |
| `boxy_sersic` | one Sérsic on generalized ellipses | 8 | boxy or discy isophotes |
| `core_sersic` | one core-Sérsic | 10 | a resolved depleted core |
| `bulge_broken_disk` | Sérsic bulge and broken exponential disc, shared centre | 14 | a disc of type I, II or III |
| `bulge_disk_bar` | Sérsic bulge, exponential disc and Ferrers bar, shared centre | 17 | a barred disc galaxy |
| `bulge_disk_ring` | Sérsic bulge, exponential disc and Gaussian ring, shared centre | 16 | a disc galaxy with a ring |
| `bulge_disk_spiral` | Sérsic bulge and exponential disc with two spiral arms, shared centre | 15 | a two-armed spiral galaxy |

The Models section of the documentation describes each model and the domain
in which its rendering is validated. Each model is one fit, and a model with
several components takes longer than a single Sérsic.

In [ ]:
models = ["single_sersic"]
# models = ["single_sersic", "double_sersic"]

## Your declarations

The advisor proposes every prior you do not declare. Declare one when you
know more than its general rule, for example the magnitude range or the
sizes of your sample: a declared prior is kept as declared and recorded as
yours.

- `model_priors`: priors by model, then by parameter, in the units of the
  configuration: fluxes in image units, lengths in pixels, angles in radians.
- `background_model`: the residual background left after preparation removes
  the sky. `constant` samples a level; `plane` also samples two gradients.
  The advisor proposes their priors from the prepared sky scale.
- `resampling_variance_factor`: for a resampled image, the factor by which
  the resampling multiplied the variance of the noise. It lets the gain
  keyword of the header be proposed as the effective gain; leave it `None`
  unless you know it.

The sampler settings change how the posterior is computed, not the
scientific problem. Nautilus returns weighted posterior samples and the
Bayesian evidence. `filepath: auto` keeps a signed checkpoint for each model
in the run directory, so an interrupted fit resumes instead of restarting.

In [ ]:
model_priors = {}
# model_priors = {
#     "single_sersic": {
#         "re": {"type": "log_uniform", "lower": 2.0, "upper": 200.0},
#     },
# }

background_model = "constant"
resampling_variance_factor = None

sampler = {
    "name": "nautilus",
    "init": {
        "seed": 42,
        "filepath": "auto",
    },
}

## Review the proposal

The proposal is built in memory, and the YAML is not written yet. The summary
lists every prior of every model with its source (`header`, `declaration`
when it rests on a declared calibration, `fixed_inputs`, `reference` for a
BayMorph default, or `user`), the residual background, the
effective gain, the PSF ceiling and the warnings. Read them and the complete
YAML before accepting it.

In [ ]:
proposal = advisor.propose_config(
    output_path=config_path,
    models=models,
    model_priors=model_priors,
    fixed_inputs=fixed_inputs,
    background_model=background_model,
    resampling_variance_factor=resampling_variance_factor,
    sampler=sampler,
    **stage_options,
)

print(proposal.summary())
print()
print(yaml.safe_dump(proposal.config, sort_keys=False))

## Write the YAML

`accept=True` records that you reviewed the proposal. It writes the YAML and,
beside it, a report, `<analysis_name>.advisor.json`, that records where every
proposed value comes from. BayMorph refuses to replace an existing YAML that
differs. To change a declaration after 02B has run, choose a new
`analysis_name`: products already written from the old YAML are never
updated.

In [ ]:
written_path = proposal.write(accept=True)

print(f"Configuration written to: {written_path}")
print(f"Advisor report written to: {proposal.report_path}")
print("Next: open 02B_Observed_Fit.ipynb.")